# PyCAM-SIMA persistent pool: reuse and fork

This Notebook keeps one MPI pool alive across cells. The persistent path launches MPI once, initializes the base once, and forks three children into pre-allocated slots. Jupyter displays each cell's execution duration, so the code contains no manual timers.

## 1. Execution paths

```text
Persistent pool                            Legacy multi-job
one launcher worker + one large mpiexec     base MPI job
├── worker A: base Actor       → slot 0     ├── child MPI job: control
├── worker B: control Actor    → slot 1     ├── child MPI job: no-kessler
├── worker C: no-kessler Actor → slot 2     └── child MPI job: warm
└── worker D: warm Actor       → slot 3

fork = MPI rank-to-rank memory copy     fork = checkpoint + new MPI startup
```

Run the persistent cells and inspect Jupyter's cell duration. The optional legacy cell performs the corresponding multi-job workload, and its cell duration includes real queue, MPI startup, checkpoint, and model costs.

## 2. Configure Dask and calculate the pool

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from pycam_sima import DaskExperimentClient

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/persistent_pool_trials' / stamp
initial_run_dir = experiment_root / 'initial-run'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

# Use allocation mode when this Jupyter server is running inside qsub -I.
execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
client = Client(
    processes=False,
    # One launcher worker plus one ModelActor worker per possible slot.
    n_workers=5,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=experiment_root / 'models',
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
resource_plan = experiments.plan_pool(
    max_concurrent_models=4,
    ranks_per_model=None,  # inherit mpi_size=24 from ModelConfig
    memory_per_model='auto',
)
{
    'pycam_sima': pycam_sima.__version__,
    'execution_mode': execution_mode,
    'run_root': str(experiment_root),
    'resource_plan': resource_plan.describe(),
}

## 3. Start the pool and base once

The pool and base are stored as normal Python variables, so later cells reuse the same live MPI processes and StatePool. Do not rerun this cell without first running the cleanup cell.

In [ ]:
pool = experiments.pool('cam-pool', resource_plan=resource_plan)

base = pool.model('base')

pool_started = pool.status
base_started = base.status
assert pool_started['mpi_launch_count'] == 1
{
    'mpi_launch_count': pool_started['mpi_launch_count'],
    'pool_mpi_launch_id': pool_started.get('pool_mpi_launch_id'),
    'pbs_job_id': pool_started.get('pbs_job_id'),
    'launcher_worker': pool.worker,
    'base_worker': base.worker,
    'base_slot': base.slot_id,
    'scheduler': pool.scheduler_status,
    'base_status': base_started,
    'slots': pool.slots,
}

## 4. A later cell reuses the same base memory

No new `mpiexec`, no model restart, and no checkpoint restore occurs here. This cell also demonstrates dynamic field creation/removal and runtime Fortran plugin loading before the fork.

In [ ]:
launches_before_reuse = pool.status['mpi_launch_count']
base.advance(steps=2)

base.fields.create(
    'experiment_tracer',
    dims=('column', 'level'),
    units='kg kg-1',
    initial=0.0,
)
base.fields.create('temporary_probe', dims=('column',), initial=1.0)
deleted_probe = base.fields.delete('temporary_probe')

installed_plugin = base.physics.install(
    source=repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=repo,
    after='kessler',
    inputs={
        'runtime_plugin_temperature': 240.0,
        'runtime_plugin_temperature_increment': 1.5,
    },
)
plugin_field = base.fields.ccpp_runtime_plugin_temperature
plugin_before = plugin_field.stats(rank=0)
base.physics.scheme('runtime_temperature_offset', group='before').run()
plugin_after = plugin_field.stats(rank=0)
assert np.isclose(plugin_after['mean'] - plugin_before['mean'], 1.5)
assert pool.status['mpi_launch_count'] == launches_before_reuse == 1
{
    'base_step': base.status.step,
    'same_mpi_launch_count': pool.status['mpi_launch_count'],
    'deleted_dynamic_field': deleted_probe['standard_name'],
    'installed_plugin': installed_plugin['name'],
    'plugin_increment': plugin_after['mean'] - plugin_before['mean'],
}

## 5. Fork three models into pre-allocated slots

The child MPI ranks already exist. Fork copies each parent rank's StatePool directly to the matching rank in an idle slot. Jupyter's duration for this cell is the fork cost.

In [ ]:
branches = base.fork(
    'control', 'no_kessler', 'warm', require_concurrent=True
)

branches.no_kessler.physics.kessler.disable()
branches.warm.fields.air_temperature += 1.0
control_temperature = branches.control.fields.air_temperature.get(rank=0)
warm_temperature = branches.warm.fields.air_temperature.get(rank=0)
assert np.array_equal(warm_temperature, np.add(control_temperature, 1.0))
assert pool.status['mpi_launch_count'] == 1
{
    'mpi_launch_count_after_fork': pool.status['mpi_launch_count'],
    'branch_statuses': branches.statuses,
    'occupied_slots': pool.slots,
}

## 6. Another cell continues the same three children

The handles and all StatePools remain live from the previous cell. Each submit call creates a Dask Future on that model's own worker. The Launcher batches ready commands for different slots into one MPI-world command.

In [ ]:
steps_before = {name: status.step for name, status in branches.statuses.items()}
control_future = branches.control.submit.advance(steps=1)
no_kessler_future = branches.no_kessler.submit.advance(steps=1)
warm_future = branches.warm.submit.advance(steps=1)
client.gather((control_future, no_kessler_future, warm_future))

# This observation is an explicit child of warm_future in Dask's graph.
warm_stats_future = branches.warm.submit.fields.air_temperature.stats(
    rank=0,
    depends_on=warm_future,
)
warm_stats = warm_stats_future.result()
steps_after = {name: status.step for name, status in branches.statuses.items()}
assert all(steps_after[name] == steps_before[name] + 1 for name in steps_before)
assert pool.status['mpi_launch_count'] == 1
{
    'steps_before': steps_before,
    'steps_after': steps_after,
    'model_workers': {
        name: model.worker for name, model in branches.items()
    },
    'model_slots': {
        name: model.slot_id for name, model in branches.items()
    },
    'warm_temperature': warm_stats,
    'mpi_launch_count': pool.status['mpi_launch_count'],
}

## 7. Optional legacy multi-job comparison

Set the flag to `True` only when the Notebook runs in `execution_mode='pbs'`. This intentionally submits one legacy base job plus three child jobs. Compare this cell's Jupyter duration with the persistent pool cells above. It is disabled by default to prevent accidental extra PBS submissions.

In [ ]:
run_legacy_comparison = False

if not run_legacy_comparison:
    legacy_result = 'skipped; set run_legacy_comparison=True for the real four-job comparison'
elif execution_mode != 'pbs':
    raise RuntimeError('legacy multi-model timing requires execution_mode="pbs"')
else:
    legacy_client = Client(
        processes=False,
        n_workers=4,
        threads_per_worker=1,
        dashboard_address=None,
    )
    legacy_experiments = DaskExperimentClient(
        legacy_client,
        config=config_path,
        initial_run_dir=initial_run_dir,
        run_root=experiment_root / 'legacy-benchmark',
        python_executable=repo / '.venv/bin/python',
        execution_mode='pbs',
    )
    try:
        with legacy_experiments.model('legacy-base') as legacy_base:
            legacy_base.advance(steps=2)
            control_plan = legacy_experiments.plan('legacy-control')
            no_kessler_plan = legacy_experiments.plan('legacy-no-kessler', experimental=True)
            no_kessler_plan.physics.kessler.disable()
            warm_plan = legacy_experiments.plan('legacy-warm')
            warm_plan.fields.edit('air_temperature', 'add', 1.0)

            legacy_children = legacy_experiments.fork_models(
                legacy_base,
                (control_plan, no_kessler_plan, warm_plan),
                close_parent=False,
            )
            with legacy_children:
                legacy_children.advance(steps=1)
                legacy_statuses = legacy_children.statuses
                legacy_child_job_ids = {
                    name: status.pbs_job_id
                    for name, status in legacy_statuses.items()
                }
            legacy_base_job_id = legacy_base.status.pbs_job_id

        legacy_result = {
            'base_pbs_job_id': legacy_base_job_id,
            'child_pbs_job_ids': legacy_child_job_ids,
        }
    finally:
        legacy_client.close()

legacy_result

## 8. Cleanup

Run this cell when finished. It closes the three children, base, pool MPI world, and Dask client in reverse creation order.

In [ ]:
branches.close()
base.close()
pool.close()
client.close()
{
    'children_closed': True,
    'base_closed': True,
    'pool_closed': True,
    'dask_client_closed': True,
}